# Direction-token location analysis

For each reasoning token, how many of its 20 jlens top-k predictions are direction words?

This is the global counterpart of `read_direction_counts` in
`telos_interp/commands/prepare_activations_for_probing/jlens_token_selection.py`, which
does the same thing per trajectory and per layer to drive token selection.

## Three fixes applied

1. **Accumulation.** `dict_counts[token] = {...}` used to sit *inside* the row loop, so a
   repeated token's counts were overwritten rather than summed — every printed per-token
   number was that token's last occurrence, not its total. Any figure taken from an
   earlier run of this notebook (`Ġdown` 5, `Ġleft` 4, `Ġat` 3, `7` 3) is a
   last-occurrence artifact and should be recomputed.

2. **Vocabulary.** Loaded `direction_tokens.json`, the older 4-way file. Now loads
   `direction_tokens_full.json` — 539 tokens (UP 172, DOWN 155, LEFT 64, RIGHT 148),
   which is what `jlens_token_selection` and the pipeline actually score against.

3. **Input.** Read a single 3.2 GB / 16.1M-row combined CSV with `pd.read_csv` +
   `df.iterrows()`. `scripts/jlens_reasoning_tokens.py` now writes **one CSV per
   trajectory** at `{activations_dir}/size{S}/{stem}/{stem}_jlens_analysis.csv`, so the
   loader below takes either a file or a directory and streams it.

   It also uses `csv.DictReader`, **not pandas**. Decoded token strings include literal
   `"NA"`, `""`, embedded newlines and commas; pandas' default NA handling silently turns
   the token `NA` into a float nan and the counts drift. Streaming also drops peak memory
   from ~16 GB to nothing.


In [ ]:
import csv
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

# csv fields can be very long (a decoded token may contain newlines); without this a
# pathological row raises _csv.Error: field larger than field limit.
csv.field_size_limit(sys.maxsize)

DATA_DIR = os.environ.get("JLENS_DATA_DIR", "/workspace/jlens")

DIRECTION_TOKENS = os.path.join(DATA_DIR, "direction_tokens_full.json")

# Either the old combined CSV or a directory of per-trajectory CSVs -- see fix 3.
JLENS_INPUT = os.path.join(DATA_DIR, "jlens_reasoning_tokens_own_jlens_1000_traj.csv")
# JLENS_INPUT = "/workspace/activations/jlens_reasoning_tokens"

TOP_K = 20
CSV_SUFFIX = "_jlens_analysis.csv"

In [ ]:
def jlens_csv_paths(input_path):
    """Every jlens CSV under `input_path`, which may be one file or a directory tree."""
    p = Path(input_path)
    if p.is_file():
        return [p]
    if p.is_dir():
        return sorted(p.glob(f"**/*{CSV_SUFFIX}"))
    raise FileNotFoundError(f"no jlens CSV at {input_path}")


def iter_jlens_rows(input_path, progress_every=1_000_000):
    """Stream rows from one or many jlens CSVs as dicts.

    csv.DictReader rather than pandas: token strings include literal "NA" and "", which
    pandas' NA handling would turn into nan. Streaming keeps memory flat over the 16M-row
    combined CSV.
    """
    n = 0
    for path in jlens_csv_paths(input_path):
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                yield row
                n += 1
                if progress_every and n % progress_every == 0:
                    print(f"  {n:,} rows", flush=True)
    print(f"  {n:,} rows total", flush=True)

In [ ]:
def count_directions(input_path, direction_data, top_k=TOP_K):
    """Accumulate per-token direction counts over the top-k j-space predictions.

    FIX 1: the per-token entry is created once via defaultdict and accumulated across
    every occurrence of that token. The previous version reassigned a fresh zeroed dict
    on each row, so only a token's final occurrence survived.

    Returns (dict_counts, totals) where dict_counts maps token -> per-direction counts.
    """
    classes = ["UP", "DOWN", "LEFT", "RIGHT"]
    lookup = {c: set(direction_data[c]) for c in classes}  # set membership, not list scan
    top_cols = [f"top_{i}" for i in range(1, top_k + 1)]

    dict_counts = defaultdict(lambda: {"UP": 0, "DOWN": 0, "LEFT": 0, "RIGHT": 0, "total": 0})
    totals = dict.fromkeys(classes, 0)

    for row in iter_jlens_rows(input_path):
        entry = dict_counts[row["token"]]
        for col in top_cols:
            value = row.get(col)
            if value is None:
                continue
            for c in classes:
                if value in lookup[c]:
                    entry[c] += 1
                    entry["total"] += 1
                    totals[c] += 1

    return dict(dict_counts), totals

In [ ]:
# --- self-test on a synthetic CSV --------------------------------------------------
#
# Verifies the accumulation fix without waiting on the real data: the token 'Ġleft'
# appears on two rows with 2 and 3 LEFT hits, so a correct implementation reports 5.
# The old code reported 3.
import tempfile

_HEADER = (
    ["size", "complexity", "run", "step", "reasoning_pos", "abs_pos", "token", "layer", "agent_action"]
    + [f"{a}_rank" for a in ("RIGHT", "LEFT", "UP", "DOWN")]
    + [f"{a}_logprob" for a in ("RIGHT", "LEFT", "UP", "DOWN")]
    + [f"top_{i}" for i in range(1, TOP_K + 1)]
)


def _row(token, layer, tops):
    pad = ["zzz"] * (TOP_K - len(tops))
    return ["11", "1.0", "0", 0, 0, 700, token, layer, "LEFT"] + [0] * 8 + list(tops) + pad


with tempfile.TemporaryDirectory() as _tmp:
    _p = Path(_tmp) / f"traj_0000{CSV_SUFFIX}"
    with open(_p, "w", newline="", encoding="utf-8") as _f:
        _w = csv.writer(_f)
        _w.writerow(_HEADER)
        _w.writerow(_row("Ġleft", 7, [" left", " links"]))  # 2 LEFT
        _w.writerow(_row("Ġleft", 8, [" left", " gauche", " izquierda"]))  # 3 LEFT
        _w.writerow(_row("NA", 7, [" up", " down"]))  # the pandas trap
    _fake = {
        "UP": [" up"],
        "DOWN": [" down"],
        "LEFT": [" left", " links", " gauche", " izquierda"],
        "RIGHT": [" right"],
    }
    _counts, _totals = count_directions(_p, _fake)

assert _counts["Ġleft"]["LEFT"] == 5, _counts["Ġleft"]  # was 3 before the fix
assert _counts["Ġleft"]["total"] == 5
assert "NA" in _counts, "the token 'NA' was lost -- pandas NA handling crept back in"
assert _counts["NA"] == {"UP": 1, "DOWN": 1, "LEFT": 0, "RIGHT": 0, "total": 2}
assert _totals == {"UP": 1, "DOWN": 1, "LEFT": 5, "RIGHT": 0}
print("self-test passed: counts accumulate across occurrences, 'NA' survives as a token")

In [ ]:
with open(DIRECTION_TOKENS, encoding="utf-8") as f:
    direction_data = json.load(f)

print({k: len(v) for k, v in direction_data.items()}, "total", sum(len(v) for v in direction_data.values()))

In [ ]:
dict_counts, totals = count_directions(JLENS_INPUT, direction_data)
print(f"\n{len(dict_counts)} distinct reasoning tokens")
print(totals)

In [ ]:
# top tokens per direction, and overall
def top_tokens(key, n=20):
    return sorted(dict_counts.items(), key=lambda kv: kv[1][key], reverse=True)[:n]


top_up_tokens = top_tokens("UP")
top_down_tokens = top_tokens("DOWN")
top_left_tokens = top_tokens("LEFT")
top_right_tokens = top_tokens("RIGHT")
top_total_tokens = top_tokens("total", 100)

for name, rows in [
    ("UP", top_up_tokens),
    ("DOWN", top_down_tokens),
    ("LEFT", top_left_tokens),
    ("RIGHT", top_right_tokens),
]:
    print(f"\n=== {name} ===")
    for tok, c in rows[:15]:
        print(f"  {tok!r:16s} {c[name]:6d}   (total {c['total']})")

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.DataFrame([{"token": t, **c} for t, c in top_total_tokens]).set_index("token")

## Reading these numbers

The top of `top_total_tokens` is expected to be the direction words themselves
(`Ġleft`, `Ġright`, `Ġdown`, `ĠMove`) and bare digits. That is the confound recorded in
`claude_session_readme.md`: a probe trained on those tokens is partly reading a decision
the model has already verbalized. In the 11×11 trajectory, 14 of the 143 analysis tokens
are tagged `("output", "analysis", "action")` and spell
`up, up, Left, LEFT, up, up, up, up, LEFT, UP, UP, UP, UP, LEFT` — the model states its
answer repeatedly mid-reasoning.

The bare digits ranking high is a different phenomenon and a more interesting one: the
tokenizer splits `row5` into `row` + `5`, so the digits carry coordinate context. No
direction vocabulary explains why a `7` should predict direction words.
